# PBS Utah historical modeling

Saved outcome-feasibility, validation-comparison and final-test results, separated from the current descriptive [EDA notebook](analysis.ipynb).

Default cells reload saved artifacts; keep rebuild switches off when reviewing results. The final test is consumed. No further model selection should use it. Pilot planning is deferred and remains in `analysis/pilot_specification.md`.

Working outcome: 12-month giving growth of at least 25% and 25 source units among individuals with positive baseline giving. Capacity 500 remains illustrative, not an approved outreach policy.

## Historical outcome feasibility and model preparation

Assessment of 12- and 24-month outcomes among individuals who had positive recorded giving in the 12 months before each cutoff. Eligibility uses only past activity. The same donor can appear at multiple cutoffs; unique positive donor counts are reported separately from donor-cutoff counts.

**Definitions remain illustrative:** aggregate giving and single-payment thresholds of 500, 1,000, 2,500, 5,000 and 10,000 source units are tested separately. A single payment need not represent a complete gift or pledge. For a 12-month giving-upgrade target, future giving must be at least 25% higher and increase by at least 25 source units. A 24-month total is not compared directly with a 12-month baseline for upgrade labeling.

Features come only from payment history available by each cutoff. Current donor-class labels, snapshot lifetime values, viewing and cultivation flags are excluded. Earlier history is left-truncated by the export start. Fiscal conventions are still unconfirmed; periods use explicit dates.

The reusable implementation is `analysis/model_readiness.py`. Default behavior reloads this validated assessment. Set `REBUILD_READINESS = True` to create a new versioned output directory.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from analysis.model_readiness import assess_model_readiness

READINESS_ROOT = Path.cwd()
if not (READINESS_ROOT / 'objectives.md').exists():
    READINESS_ROOT = READINESS_ROOT.parent
READINESS_RUN = READINESS_ROOT / 'data/model_readiness/20261010T050128Z_d90f0850'
REBUILD_READINESS = False
if REBUILD_READINESS:
    readiness = assess_model_readiness(
        READINESS_ROOT, READINESS_ROOT / 'data/marts/20261010T043152Z_873c7300')
else:
    assert (READINESS_RUN / '_SUCCESS').exists()
    readiness = {
        'output': READINESS_RUN,
        'manifest': json.loads((READINESS_RUN / 'manifest.json').read_text()),
        'counts': pd.read_csv(READINESS_RUN / 'outcome_counts.csv'),
        'split_summary': pd.read_csv(READINESS_RUN / 'split_summary.csv'),
    }
print('Assessment outputs:', readiness['output'])
print('Historical donor-cutoff/horizon rows:', readiness['manifest']['rows'])
print('Timing and cohort checks passed:', readiness['manifest']['validation_checks'])
print('Eligible population:', readiness['manifest']['population'])


In [ ]:
sensitivity = readiness['split_summary']
chosen_targets = [
    'target_retained', 'target_upgrade_25pct_and_25units',
    'target_total_ge_500', 'target_total_ge_1000',
    'target_total_ge_2500', 'target_total_ge_5000',
]
twelve = sensitivity.loc[sensitivity.horizon_months.eq(12) & sensitivity.target.isin(chosen_targets)]
print('12-month positive outcome counts, not model performance:')
print(twelve.pivot(index='target', columns='split', values='positive_outcomes')[['train','validation','test']].to_string())
print('\n12-month training: unique positive donors vs repeated donor-cutoff outcomes:')
print(twelve.loc[twelve.split.eq('train'), ['target','positive_outcomes','unique_positive_donors','positive_fraction']].to_string(index=False))
print('\n24-month aggregate-giving feasibility (no separate chronological validation cohort):')
print(sensitivity.loc[sensitivity.horizon_months.eq(24) & sensitivity.split.isin(['train','test']) & sensitivity.target.isin(['target_total_ge_1000','target_total_ge_2500','target_total_ge_5000']), ['split','target','positive_outcomes','unique_positive_donors']].to_string(index=False))


### Chronological evaluation plan

For 12-month outcomes: training cutoffs end June 2021–June 2023 (outcomes through June 2024); validation cutoff is June 2024 (outcomes through June 2025); test cutoff is June 2025 (outcomes through June 2026). All training labels are known by the validation cutoff and all validation labels by the test cutoff. A donor may appear across periods because this evaluates future behavior of known donors, not generalization exclusively to unseen donors.

For 24-month outcomes, the available six years do not support a separate train/validation/test arrangement with at least 12 months of feature history and labels known before the next prediction cutoff. A train/test split is prepared; intervening overlapping cohorts are explicitly unused for the final split.

These counts were inspected before model fitting. The later sections record the completed validation comparison and final test evaluation. The final test is now consumed; retain that separation for future development.

In [ ]:
import pyarrow.parquet as pq
cohort_path = readiness['output'] / 'historical_donor_cohorts.parquet'
assert pq.ParquetFile(cohort_path).metadata.num_rows == readiness['manifest']['rows']
historical_cohorts = pd.read_parquet(cohort_path)
predictor_columns = readiness['manifest']['features']
assert not any(col.startswith('target_') or col.startswith('outcome_') for col in predictor_columns)
assert all(col in historical_cohorts.columns for col in predictor_columns)
cohorts12 = historical_cohorts.loc[historical_cohorts.horizon_months.eq(12)]
train12 = cohorts12.loc[cohorts12.suggested_split.eq('train')]
validation12 = cohorts12.loc[cohorts12.suggested_split.eq('validation')]
test12 = cohorts12.loc[cohorts12.suggested_split.eq('test')]
assert train12.outcome_end.max() <= validation12.cutoff_date.min()
assert validation12.outcome_end.max() <= test12.cutoff_date.min()
assert historical_cohorts.last_observed_payment_date.le(historical_cohorts.cutoff_date).all()
print('Saved cohort row count, predictor whitelist, and temporal split checks passed.')
print('12-month train / validation / test rows:', len(train12), len(validation12), len(test12))
print('Predictor columns:', ', '.join(predictor_columns))
print('Target candidate: target_upgrade_25pct_and_25units (illustrative definition).')


### Recommendation and next implementation step

**Recommend 12-month giving-upgrade prediction as the first modeling candidate**, with retention as a complementary task. The illustrative upgrade definition yields 4,862 training positives across 4,642 distinct donors, 1,563 validation positives and 1,910 test positives. It is more directly connected to increased giving than retention and has substantially more examples than the higher major-giving thresholds.

The annual aggregate >=1,000 rule has only 21 training positives across 20 donors, 4 validation positives and 8 test positives. The >=2,500 rule has 2 training positives and none in validation. No eligible cohort has a >=5,000 annual aggregate positive. These counts do not support a reliable first high-threshold major-gift classifier with these data. A 24-month >=1,000 rule gives 100 training outcomes across 77 donors, but the longer horizon limits chronological validation and is a different business outcome.

The subsequent sections record the completed first model comparison and frozen test evaluation. The target remains a working definition rather than a confirmed major-gift classification. Historical propensity does not estimate additional giving caused by outreach.

## First upgrade-ranking model and simple baselines

Working outcome: at least **25% and 25 source units more giving in the next 12 months**, among individuals with positive giving in the preceding 12 months. This is an upgrade-propensity task, not a major-gift, wealth, or outreach-effect model.

Compared five simple rules with fixed L2 logistic regression and a depth-3 decision tree. Payment-only features are log-transformed amounts/counts/recency/observed tenure and clipped historical growth. Median imputation, missingness indicators and scaling are fitted on training only. Training has 95,025 donor-cutoff rows; validation has 32,731 individuals and 1,563 upgrades. The final test cohort is excluded from the model-comparison read.

Capacities of 100, 250, 500 and 1,000 are illustrative. Selection uses the top-500 validation list, with average precision as a tiebreaker. Rules and tree leaves can produce large ties: tie-adjusted metrics average over equally scored candidates at the boundary rather than benefiting from arbitrary donor-ID ordering. Raw deterministic-list metrics are retained separately.

Implementation: `analysis/upgrade_model.py`. Saved models expect `feature_matrix(raw_historical_features)`, not raw columns directly. Default behavior reloads the validated comparison; set `REBUILD_UPGRADE_MODELS = True` to create a new run.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from analysis.upgrade_model import assess_upgrade_models, plot_upgrade_comparison, feature_matrix

MODEL_ROOT = Path.cwd()
if not (MODEL_ROOT / 'objectives.md').exists():
    MODEL_ROOT = MODEL_ROOT.parent
MODEL_RUN = MODEL_ROOT / 'data/models/20261011T045412Z_a1281ec2'
REBUILD_UPGRADE_MODELS = False
if REBUILD_UPGRADE_MODELS:
    upgrade_models = assess_upgrade_models(
        MODEL_ROOT, MODEL_ROOT / 'data/model_readiness/20261010T050128Z_d90f0850')
else:
    assert (MODEL_RUN / '_SUCCESS').exists()
    upgrade_models = {
        'output': MODEL_RUN,
        'manifest': json.loads((MODEL_RUN / 'manifest.json').read_text()),
        'metrics': pd.read_csv(MODEL_RUN / 'validation_capacity_metrics.csv'),
        'overall': pd.read_csv(MODEL_RUN / 'validation_overall_metrics.csv'),
        'coefficients': pd.read_csv(MODEL_RUN / 'logistic_coefficients.csv'),
    }
print('Saved comparison:', upgrade_models['output'])
print('Selected at illustrative capacity 500:', upgrade_models['manifest']['selected_method'])
print('Model-comparison checks passed:', upgrade_models['manifest']['validation_checks'])
print('Final test scored:', upgrade_models['manifest']['test_scored'])


In [ ]:
capacity_comparison = upgrade_models['metrics'].loc[
    upgrade_models['metrics'].capacity.eq(upgrade_models['manifest']['primary_capacity'])
].sort_values('tie_adjusted_upgrades', ascending=False).copy()
capacity_comparison['precision_percent'] = capacity_comparison.tie_adjusted_precision * 100
print('Historical validation, illustrative top-500 lists:')
print(capacity_comparison[['method','tie_adjusted_upgrades','precision_percent','tie_adjusted_lift','ties_at_boundary']].to_string(index=False))
print('\nExpected upgrades from random selection:',
      round(upgrade_models['manifest']['validation_prevalence'] * 500, 2))
print('\nOverall validation ranking metrics:')
print(upgrade_models['overall'].to_string(index=False))
print('\nTie-adjusted precision (%) by capacity:')
print((upgrade_models['metrics'].pivot(index='method', columns='capacity',
      values='tie_adjusted_precision') * 100).round(2).to_string())


In [ ]:
print('Logistic coefficients: standardized log features, not causal effects.')
print(upgrade_models['coefficients'].sort_values('standardized_coefficient').to_string(index=False))

# Predictor generation must not respond to outcome/target columns.
readiness_source = MODEL_ROOT / upgrade_models['manifest']['source_readiness_run']
sample_validation = pd.read_parquet(
    readiness_source / 'historical_donor_cohorts.parquet',
    filters=[('horizon_months','=',12), ('suggested_split','=','validation')]
).head(50)
original_predictors = feature_matrix(sample_validation)
changed_outcomes = sample_validation.copy()
for col in changed_outcomes:
    if col.startswith('target_'):
        changed_outcomes[col] = False
    if col.startswith('outcome_') and col.endswith('_hundredths'):
        changed_outcomes[col] = 999999999
pd.testing.assert_frame_equal(original_predictors, feature_matrix(changed_outcomes))
assert upgrade_models['manifest']['test_scored'] is False
print('\nPredictor invariance to modified outcomes/targets passed; final test remains reserved.')


In [ ]:
import matplotlib.pyplot as plt
upgrade_figure, upgrade_chart_path = plot_upgrade_comparison(upgrade_models)
plt.show()
print('Chart saved:', upgrade_chart_path)


### What the comparison shows

At the illustrative capacity of 500, logistic regression finds **77** historical validation upgrades (**15.4% precision**, **3.22× random lift**). The depth-3 tree has **67.6** tie-adjusted expected upgrades; the strongest simple rule, lower prior giving, has **62.9**; random selection expects **23.9**. Fractional counts for tied rules are averages over possible choices within ties, not literal fractional donors.

Logistic regression has average precision **0.109**, versus **0.099** for the strongest simple rule and an overall validation upgrade rate of **0.0478**. Its top-500 list captures **4.9%** of all validation upgrades. Most selected donors did not upgrade; this is a prioritization signal, not a guarantee.

The best approach depends on capacity: the tree performs better at 100, while logistic regression leads at 250, 500 and 1,000 in this validation comparison. Staff capacity is not yet known. The apparent improvement is a validation result used for selection, not a final test result or proof of additional gifts caused by outreach.

Lower prior giving is a strong signal partly because a percentage-based upgrade is easier from a smaller base. Do not describe this ranking as finding the wealthiest donors or maximizing gift revenue. Synthetic source data and incomplete historical coverage limit operational use. Scores have not undergone separate probability calibration.

### Evaluation status

The frozen chronological test evaluation is completed in the next section. This comparison remains the validation-stage record, not the final performance estimate.

## Final chronological evaluation of the frozen policy

This completes the next step from the preceding comparison. The working target (25% and 25 source units of giving growth), saved training-only logistic regression, strongest validation-selected simple benchmark, and illustrative capacity 500 were frozen before test loading. Secondary capacities were already specified in the comparison. No refitting, feature selection or hyperparameter tuning used the test data.

Prediction cutoff: June 30, 2025; outcomes through June 30, 2026. Implementation: `analysis/final_evaluation.py`. The frozen policy, model/data hashes and evaluation checks are saved with the results. The earlier comparison manifest correctly describes that earlier run as test-unscored; this separate evaluation records the newly consumed test.

Default cells reload this saved evaluation rather than evaluating the test again.

In [ ]:
from pathlib import Path
import json
import pandas as pd

FINAL_ROOT = Path.cwd()
if not (FINAL_ROOT / 'objectives.md').exists():
    FINAL_ROOT = FINAL_ROOT.parent
FINAL_RUN = FINAL_ROOT / 'data/final_evaluation/20261011T055348Z_689b5d06'
assert (FINAL_RUN / '_SUCCESS').exists()
final_manifest = json.loads((FINAL_RUN / 'manifest.json').read_text())
final_policy = json.loads((FINAL_RUN / 'frozen_policy.json').read_text())
final_metrics = pd.read_csv(FINAL_RUN / 'test_capacity_metrics.csv')
final_overall = pd.read_csv(FINAL_RUN / 'test_overall_metrics.csv')
print('Evaluation:', FINAL_RUN)
print('Test donors / upgrades:', final_manifest['test_rows'], final_manifest['test_positive_outcomes'])
print('Prediction cutoff / outcome end:', final_manifest['cutoff_date'], final_manifest['outcome_end'])
print('Frozen method / benchmark / primary capacity:', final_policy['selected_method'], final_policy['benchmark'], final_policy['primary_capacity'])
print('Checks passed:', final_manifest['validation_checks'])

In [ ]:
primary_test = final_metrics.loc[final_metrics.capacity.eq(final_policy['primary_capacity'])]
print('Final chronological test, frozen top-500 policy:')
print(primary_test.to_string(index=False))
print('\nOverall test ranking metrics:')
print(final_overall.to_string(index=False))
selected_result = primary_test.loc[primary_test.method.eq(final_policy['selected_method'])].iloc[0]
benchmark_result = primary_test.loc[primary_test.method.eq(final_policy['benchmark'])].iloc[0]
print('\nAdditional historical upgrades found relative to tie-adjusted benchmark:', round(selected_result.tie_adjusted_upgrades - benchmark_result.tie_adjusted_upgrades, 2))
print('Test cohort is now consumed. Do not tune models or capacity using these results.')

### Interpretation and next decision

The frozen logistic model identifies **87 upgrades among 500 test donors (17.4% precision; 2.99× random lift)**, versus **61.4 tie-adjusted expected upgrades (12.28%)** for lower prior giving and **29.1** expected under random selection. Its top-500 list captures **4.55%** of all test upgrades. Average precision is **0.1174**, versus **0.0977** for the benchmark. Most selected donors did not upgrade. This is descriptive evidence of historical ranking value, not a significance test, calibrated probability claim or outreach effect.

The test cohort is now consumed. Keep these results as the final evaluation of this frozen policy; later feature/model choices require a new untouched period or evaluation design. Capacity 500 and the upgrade definition remain illustrative rather than approved business policies. Source data contains synthetic placeholders.

Operational pilot work is deferred. Its draft is retained in `analysis/pilot_specification.md`; current assignments, permissions and staff costs are not prerequisites for continuing descriptive EDA. No current donor scores or outreach were created.